In [21]:
import pandas as pd
import nltk
import ssl

import certifi

In [22]:
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

In [23]:
nltk.download("popular")
nltk.download('punkt')
nltk.download("punkt_tab")

[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     /Users/infic/nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     /Users/infic/nltk_data...
[nltk_data]    |   Package gazetteers is already up-to-date!
[nltk_data]    | Downloading package genesis to
[nltk_data]    |     /Users/infic/nltk_data...
[nltk_data]    |   Package genesis is already up-to-date!
[nltk_data]    | Downloading package gutenberg to
[nltk_data]    |     /Users/infic/nltk_data...
[nltk_data]    |   Package gutenberg is already up-to-date!
[nltk_data]    | Downloading package inaugural to
[nltk_data]    |     /Users/infic/nltk_data...
[nltk_data]    |   Package inaugural is already up-to-date!
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]    |     /Users/infic/nltk_data...
[nltk_data]    |   Package movie_reviews is already up-to

True

In [24]:
dataset = pd.read_csv("../dataset/output.csv")

dataset.head()

,text,label
0,А Бог с вами!\r\nБудьте овцами!\r\nХодите стад...,cvetaeva
1,А во лбу моем – знай! –\r\nЗвезды горят.\r\nВ ...,cvetaeva
2,"А над равниной –\r\nКрик лебединый.\r\nМатерь,...",cvetaeva
3,А что если кудри в плат\r\nУпрячу – что вьются...,cvetaeva
4,"Август – астры,\r\nАвгуст – звезды.\r\nАвгуст ...",cvetaeva


In [25]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(dataset["text"], dataset["label"], test_size=0.15, random_state=42, stratify=dataset["label"])

In [26]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import re

stop_words = set(stopwords.words("russian"))
stemmer = SnowballStemmer(language="russian")

documents = []

def get_correct_tokens(row: pd.Series):
    tokens = word_tokenize(row["text"])
    tokens = [re.sub(r'[^\w\s]', '', token) for token in tokens if re.sub(r'[^\w\s]', '', token)]

    lemmatized_tokens = [stemmer.stem(word) for word in tokens]

    clean_tokens = [token for token in lemmatized_tokens if token not in stop_words]

    return clean_tokens, row["label"]

for _, row in dataset.iterrows():
    tokens, author = get_correct_tokens(row)
    documents.append(" ".join(tokens))



In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, confusion_matrix

regressor = LinearSVC(random_state=42)
vectorizer = TfidfVectorizer().fit(documents)

vectorized_train_dataset = vectorizer.transform(X_train)

regressor = regressor.fit(vectorized_train_dataset, y_train)

test_predicts = regressor.predict(vectorizer.transform(X_test))

results = f1_score(y_test, test_predicts, average="macro")

results

0.836775745510348

In [28]:
confusion_matrix(y_test, test_predicts)

array([[87,  4,  0],
       [ 8, 46,  4],
       [ 2,  6, 21]])